# AI-Based Pothole Recognition Using UAVs
## Google Colab 50-Epoch GPU Training Notebook (YOLOv11 & YOLOv8)

**Project:** Automated UAV Road Inspection & Municipal Priority Dispatch  
**Course:** BCSE306L – Artificial Intelligence (VIT)  
**Faculty:** Dr. Vijayaprabakaran K  
**Students:** Ajay Kumaar (24BRS1287) & Hariaswath (24BRS1290)  

---
### Instructions
1. In Google Colab, go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Drag and drop `pothole_dataset.zip` into the Colab file explorer on the left.
3. Run all cells. In ~4 minutes, it trains 50 epochs and automatically downloads your trained `best.pt` model!

In [ ]:
# Step 1: Verify Free T4 GPU Acceleration
!nvidia-smi

# Step 2: Install Ultralytics and dependencies
!pip install -q ultralytics opencv-python pillow matplotlib pyyaml tqdm

# Step 3: Unzip the uploaded dataset
!unzip -q -o pothole_dataset.zip
print("Dataset successfully unzipped!")

In [ ]:
# Step 4: Verify PyTorch CUDA Support
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Accelerated GPU Device: {torch.cuda.get_device_name(0)}")

---
### Step 5: Train YOLOv11 for 50 Epochs (Proposed SOTA)
Trains with decoupled anchor-free head on 640x640 images. Takes ~3-4 minutes on T4 GPU.

In [ ]:
from ultralytics import YOLO

# Initialize YOLOv11 model
model_yolo11 = YOLO('yolo11n.pt')

# Train on real dataset for 50 full epochs
results_yolo11 = model_yolo11.train(
    data='dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='runs/colab',
    name='yolo11_sota_50epochs',
    plots=True,
    verbose=True
)

---
### Step 6: Validate Precision, Recall & mAP Metrics

In [ ]:
val_results = model_yolo11.val()
print("=" * 60)
print("FINAL 50-EPOCH MODEL EVALUATION METRICS:")
print(f"Precision:  {val_results.results_dict.get('metrics/precision(B)', 0):.4f}")
print(f"Recall:     {val_results.results_dict.get('metrics/recall(B)', 0):.4f}")
print(f"mAP@50:     {val_results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"mAP@50-95:  {val_results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")
print("=" * 60)

---
### Step 7: Download the Trained Model Checkpoint (`best.pt`)
This automatically triggers a browser download for the trained `best.pt` weights file.

In [ ]:
from google.colab import files
print("Downloading best.pt to your computer...")
files.download('runs/colab/yolo11_sota_50epochs/weights/best.pt')